# DeepFormer GPU Demo Notebook



## 0. 进入项目根目录

先确认工作目录是项目根目录。

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path('/mnt/public5/genomebench/ningyuan/projects/deepformer-refactored')
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('PWD =', Path.cwd())

PROJECT_ROOT = /mnt/public5/genomebench/ningyuan/projects/deepformer-refactored
PWD = /mnt/public5/genomebench/ningyuan/projects/deepformer-refactored


## 1. 导入依赖并配置 Python 路径

In [2]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from models.deepformer import DeepFormer
from utils.deepsea_demo_npz_dataset import DeepSEADemoNPZDataset

print('Imports OK')

Imports OK


## 2. GPU 检查与稳定性设置

In [3]:
import gc
import torch

print("torch version =", torch.__version__)
print("cuda available =", torch.cuda.is_available())
print("cuda device count =", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))


gpu_id = 2
device = torch.device(f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.cuda.set_device(gpu_id)
    torch.cuda.empty_cache()
    gc.collect()


torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("device =", device)

torch version = 2.5.1
cuda available = True
cuda device count = 4
GPU 0: NVIDIA GeForce RTX 4090
GPU 1: NVIDIA GeForce RTX 4090
GPU 2: NVIDIA GeForce RTX 4090
GPU 3: NVIDIA GeForce RTX 4090
device = cuda:2


In [ ]:
!nvidia-smi

In [4]:
torch.cuda.empty_cache()
gc.collect()

x_test = torch.randn(1, 4, 1000, device=device)
conv_test = torch.nn.Conv1d(4, 8, kernel_size=7).to(device)

with torch.no_grad():
    y_test = conv_test(x_test)

print("minimal conv test ok:", y_test.shape)

del x_test, conv_test, y_test
torch.cuda.empty_cache()
gc.collect()

minimal conv test ok: torch.Size([1, 8, 994])


0

## 3. 定义辅助函数

In [5]:
def normalize_output(output):
    if isinstance(output, torch.Tensor):
        return output
    if isinstance(output, (list, tuple)) and len(output) > 0:
        if isinstance(output[0], torch.Tensor):
            return output[0]
    raise TypeError(f'Unsupported model output type: {type(output)}')


def binary_accuracy(preds: torch.Tensor, targets: torch.Tensor, threshold: float = 0.5) -> float:
    pred_bin = (preds >= threshold).float()
    correct = (pred_bin == targets).float()
    return float(correct.mean().item())


def count_parameters(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Helper functions ready')

Helper functions ready


## 4. 加载 demo subset 数据

In [6]:
train_npz = PROJECT_ROOT / 'data' / 'raw_deepsea' / 'demo_subset' / 'train_demo_256.npz'
valid_npz = PROJECT_ROOT / 'data' / 'raw_deepsea' / 'demo_subset' / 'valid_demo_256.npz'
test_npz  = PROJECT_ROOT / 'data' / 'raw_deepsea' / 'demo_subset' / 'test_demo_256.npz'

print(train_npz)
print(valid_npz)
print(test_npz)

train_ds = DeepSEADemoNPZDataset(str(train_npz))
valid_ds = DeepSEADemoNPZDataset(str(valid_npz))
test_ds  = DeepSEADemoNPZDataset(str(test_npz))

print('train len =', len(train_ds))
print('valid len =', len(valid_ds))
print('test  len =', len(test_ds))

x0, y0 = train_ds[0]
print('single x shape =', tuple(x0.shape))
print('single y shape =', tuple(y0.shape))
print('single x dtype =', x0.dtype)
print('single y dtype =', y0.dtype)

/mnt/public5/genomebench/ningyuan/projects/deepformer-refactored/data/raw_deepsea/demo_subset/train_demo_256.npz
/mnt/public5/genomebench/ningyuan/projects/deepformer-refactored/data/raw_deepsea/demo_subset/valid_demo_256.npz
/mnt/public5/genomebench/ningyuan/projects/deepformer-refactored/data/raw_deepsea/demo_subset/test_demo_256.npz
train len = 256
valid len = 256
test  len = 256
single x shape = (4, 1000)
single y shape = (919,)
single x dtype = torch.float32
single y dtype = torch.float32


## 5. 构造 DataLoader

In [7]:
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print('batch x shape =', tuple(xb.shape))
print('batch y shape =', tuple(yb.shape))

batch x shape = (1, 4, 1000)
batch y shape = (1, 919)


## 6. 构造 DeepFormer 模型

In [8]:
model = DeepFormer(sequence_length=1000, n_targets=919).to(device)
print(model.__class__.__name__)
print('trainable params =', count_parameters(model))

DeepFormer
trainable params = 54640559


## 7. GPU 前向传播测试

In [9]:
torch.cuda.empty_cache()
gc.collect()

model.eval()
xb, yb = next(iter(train_loader))

xb = xb.to(device=device, dtype=torch.float32).contiguous()
yb = yb.to(device=device, dtype=torch.float32).contiguous()

with torch.no_grad():
    out = normalize_output(model(xb))

print('input shape  =', tuple(xb.shape))
print('label shape  =', tuple(yb.shape))
print('output shape =', tuple(out.shape))
print('output dtype =', out.dtype)
print('output mean  =', float(out.mean().item()))
print('output std   =', float(out.std().item()))
print('gpu memory allocated (MB) =', torch.cuda.memory_allocated() / 1024 / 1024)

input shape  = (1, 4, 1000)
label shape  = (1, 919)
output shape = (1, 919)
output dtype = torch.float32
output mean  = 0.49979737401008606
output std   = 0.0075803399085998535
gpu memory allocated (MB) = 216.6083984375


## 8. tiny GPU training

In [10]:
model = DeepFormer(sequence_length=1000, n_targets=919).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

max_steps = 5
model.train()

train_log = []

for step, (xb, yb) in enumerate(train_loader, start=1):
    xb = xb.to(device)
    yb = yb.to(device)

    optimizer.zero_grad()
    out = normalize_output(model(xb))
    loss = criterion(out, yb)
    loss.backward()
    optimizer.step()

    log_line = f'STEP {step} | TRAIN LOSS {float(loss.item()):.6f}'
    train_log.append(log_line)
    print(log_line)

    if step >= max_steps:
        break

STEP 1 | TRAIN LOSS 0.693604
STEP 2 | TRAIN LOSS 0.424599
STEP 3 | TRAIN LOSS 0.192713
STEP 4 | TRAIN LOSS 0.381363
STEP 5 | TRAIN LOSS 0.026850


## 9. GPU 上做一个简单验证

In [11]:
model.eval()
val_losses = []
val_accs = []

with torch.no_grad():
    for xb, yb in valid_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        out = normalize_output(model(xb))
        loss = criterion(out, yb)
        acc = binary_accuracy(out, yb)
        val_losses.append(float(loss.item()))
        val_accs.append(float(acc))

print('avg valid loss =', sum(val_losses) / len(val_losses))
print('avg valid acc  =', sum(val_accs) / len(val_accs))

avg valid loss = 0.31043858664270374
avg valid acc  = 0.9676618394441903


## 10. 保存 GPU demo checkpoint 和日志

In [12]:
results_dir = PROJECT_ROOT / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

ckpt_path = results_dir / 'deepformer_gpu_demo_model.pt'
torch.save(model.state_dict(), ckpt_path)
print('checkpoint saved to:', ckpt_path)

log_path = results_dir / 'deepformer_gpu_demo_train_log.txt'
with open(log_path, 'w', encoding='utf-8') as f:
    f.write('DeepFormer GPU demo notebook log\n\n')
    for line in train_log:
        f.write(line + '\n')
    f.write('\n')
    f.write(f'avg valid loss = {sum(val_losses) / len(val_losses)}\n')
    f.write(f'avg valid acc = {sum(val_accs) / len(val_accs)}\n')
    f.write(f'checkpoint = {ckpt_path}\n')

print('log saved to:', log_path)

checkpoint saved to: /mnt/public5/genomebench/ningyuan/projects/deepformer-refactored/results/deepformer_gpu_demo_model.pt
log saved to: /mnt/public5/genomebench/ningyuan/projects/deepformer-refactored/results/deepformer_gpu_demo_train_log.txt
